# Hierarchical clustering na prática: agrupando clientes
Envie o arquivo `compras.csv` quando a célula 2 pedir. É o mesmo arquivo da aula de K-Means.

In [ ]:
# 1. Importar bibliotecas
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.preprocessing import StandardScaler
from google.colab import files

In [ ]:
# 2. Enviar o arquivo compras.csv
files.upload()
df = pd.read_csv("compras.csv", parse_dates=["InvoiceDate"])
df.head()

In [ ]:
# 3. Montar o resumo por cliente (uma linha por cliente)
df = df.dropna(subset=["CustomerID"])
df["Total"] = df["Quantity"] * df["UnitPrice"]

cli = df.groupby("CustomerID").agg(
    Frequencia=("InvoiceNo", "nunique"),
    Gasto=("Total", "sum"))
cli.head()

In [ ]:
# 4. Normalizar (todas as variáveis com o mesmo peso)
X = StandardScaler().fit_transform(cli)

In [ ]:
# 5. Linkage e dendrograma
Z = linkage(X, method="ward")   # single, complete, average ou ward

plt.figure(figsize=(10, 4))
dendrogram(Z, truncate_mode="lastp", p=20)
plt.xlabel("Cliente ou grupo (o número entre parênteses é a quantidade de clientes)")
plt.ylabel("Distância")
plt.title("Dendrograma dos clientes")
plt.show()

In [ ]:
# 6. Escolher o corte: uma linha horizontal no dendrograma
plt.figure(figsize=(10, 4))
dendrogram(Z, truncate_mode="lastp", p=20)
plt.axhline(y=3.5, color="red", linestyle="--")   # altura do corte
plt.ylabel("Distância")
plt.title("Corte em 3,5 = 3 grupos")
plt.show()

In [ ]:
# 7. Aplicar o agrupamento hierárquico com 3 grupos
modelo = AgglomerativeClustering(n_clusters=3, linkage="ward")
cli["Grupo"] = modelo.fit_predict(X)

cli["Grupo"].value_counts().sort_index()

In [ ]:
# 8. Interpretar: média de cada grupo
cli.groupby("Grupo").agg(
    clientes=("Gasto", "count"),
    frequencia_media=("Frequencia", "mean"),
    gasto_medio=("Gasto", "mean")).round(1)

In [ ]:
# 9. Visualizar os grupos
plt.scatter(cli["Frequencia"], cli["Gasto"], c=cli["Grupo"], cmap="viridis")
plt.xlabel("Frequência (nº de compras)")
plt.ylabel("Gasto total (R$)")
plt.title("Clientes agrupados (Ward, 3 grupos)")
plt.show()

In [ ]:
# 10. Comparar os tipos de linkage (3 grupos em cada)
for metodo in ["single", "complete", "average", "ward"]:
    rotulos = AgglomerativeClustering(n_clusters=3, linkage=metodo).fit_predict(X)
    tamanhos = pd.Series(rotulos).value_counts().sort_values(ascending=False).tolist()
    print(f"{metodo:9s} -> clientes por grupo: {tamanhos}")

In [ ]:
# 11. Cortar por distância em vez de fixar o número de grupos
modelo = AgglomerativeClustering(n_clusters=None, distance_threshold=3.5, linkage="ward")
modelo.fit(X)
print("Grupos encontrados:", modelo.n_clusters_)

In [ ]:
# 12. Comparar com o K-Means (K = 3)
cli["KMeans"] = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X)
pd.crosstab(cli["Grupo"], cli["KMeans"])